# 최대수요전력 예측과 피크 관리 — 전체 실행 노트북

이 노트북을 **위에서부터 차례로 실행하면** 결과보고서의 숫자를 다시 만든다. 자세한 설명은 `README.md`에 있다.

| 순서 | 단계 | 하는 일 | 기본 실행 시간 (CPU) |
|---|---|---|---|
| 0 | 환경 확인 | 버전 출력, 필요한 파일 확인 | 수 초 |
| 1 | 전처리 | 원자료 + 기상청 자료 → 15분 확정 데이터. 기존 확정 파일과 해시 대조 | 약 10초 |
| 2 | 딥러닝 | 저장된 학습 결과로 검증·시험·제출 예측 재현(`main.py --resume`), 마진 단계 입력 내보내기 | 약 2분 |
| 3 | 트리 모델 | 저장 예측을 종합 오차 지표 C로 채점 | 수 초 |
| 4 | 마진·경보 | R1 마진, 위험한 날·위험 시간 경보, 9월 채점 | 약 1분 |
| 5 | 피크 분석·방안 | 피크 사건 분석, 목표선, 부하 이동, 요금 환산 | 약 1분 |
| 6 | 요약 | 보고서 핵심 숫자를 표 번호와 함께 출력하고 보고서 값과 대조 | 수 초 |

**무거운 재학습은 기본으로 꺼져 있다.** 딥러닝 전체 재학습(GPU에서 수 시간)과 트리 모델 재학습(CPU 약 2.5시간)은 2·3단계의 옵션 칸에서 켤 수 있다.

- 미래 생산량·기상은 실제값을 계획·예보로 안다고 가정했다(다앎 가정).
- 9월(1~14일)은 모델·마진 고르기와 조정에 쓰지 않은 평가 기간이다.

## 0. 환경 확인

In [1]:
import hashlib, os, re, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "paths.py").exists(), "이 노트북은 제출 폴더 최상위(paths.py가 있는 곳)에서 실행한다"
sys.path.insert(0, str(ROOT))
import paths as P

PY = sys.executable
print("Python", sys.version.split()[0], "| pandas", pd.__version__)
for mod in ["numpy", "scipy", "torch"]:
    try:
        print(f"{mod:6s}", __import__(mod).__version__)
    except ImportError:
        print(f"{mod:6s} 없음 (2단계 딥러닝에 torch 필요)")

need = [P.RAW_ZIP, P.KMA, P.MODEL_INPUT, P.DL / "main.py", P.TREE_PRED / "1h.csv"]
for p in need:
    print("있음" if p.exists() else "없음", p.relative_to(ROOT))
assert all(p.exists() for p in need), "필요한 파일이 없다. README의 폴더 구성을 확인한다"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def run(script, cwd, args=(), show=6, cpu=False):
    """코드 파일 하나를 단독 실행하고, 마지막 몇 줄만 보여 준다(컴퓨터 경로는 가린다). cpu=True면 GPU를 끈다."""
    t = time.time()
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="") if cpu else None
    r = subprocess.run([PY, "-X", "utf8", script, *args], cwd=cwd, capture_output=True, text=True,
                       encoding="utf-8", errors="replace", env=env)
    out = (r.stdout + r.stderr).replace(str(ROOT), ".")
    out = re.sub(r"[A-Za-z]:\\[^\s'\"]*", "<경로>", out)
    lines = [s for s in out.splitlines() if s.strip()]
    print(f"[{'완료' if r.returncode == 0 else '실패'}] {script} ({time.time() - t:.0f}초)")
    for s in lines[-show:] if show else []:
        print("    " + s[:160])
    if r.returncode != 0:
        raise RuntimeError(f"{script} 실행 실패")

Python 3.12.10 | pandas 2.3.3
numpy  2.4.6
scipy  1.18.0


torch  2.11.0+cu128
있음 data\원본\5. 자원 최적화 AI 데이터셋.zip
있음 data\외부\ulsan_20210101-20210914.xls
있음 2_딥러닝\data\timeseries_15min_v1_previous_week.csv
있음 2_딥러닝\main.py
있음 3_트리모델\저장예측\1h.csv


## 1. 전처리

원본 zip(주최측 제공)과 기상청 울산 시간자료(외부 데이터)에서 15분 확정 데이터를 만든다. 단계별 코드는 `1_전처리/5_코드/`에 있고, 결과는 `1_전처리/3_최종데이터/`에 저장된다.

| 코드 | 하는 일 |
|---|---|
| 1.1 | 7/13·7/15의 깨진 시간 칸을 줄 순서대로 복원하고 인건비를 다시 계산 |
| 1.2 | 시간 단위 → 15분 단위 전개 |
| 1.3 | 기상 칸을 기상청 울산(152) 원자료로 교체 |
| 1.4 | 요일·공휴일·주야 칸 |
| 1.5 | 전력 0 구간(74칸) 표시와 1주 전 값 대체 |
| 1.6 | 7/13·7/15 생산량 대체 |
| 1.7 | 전기요금(계절) 칸의 겨울·봄가을 값 교환 |
| 1.8 | 확정 파일 저장(v1·v2, label·onehot) |

In [2]:
PRE_CODE = ROOT / "1_전처리" / "5_코드"
for s in ["1.1_시간복원.py", "1.2_15분변환.py", "1.3_기상교체.py", "1.4_달력주야.py",
          "1.5_전력0처리.py", "1.6_생산량.py", "1.7_전기요금.py", "1.8_파일저장.py"]:
    run(s, PRE_CODE, show=2)

# 보고서 작성에 쓴 확정 파일과 해시(SHA-256)가 같은지 확인한다
EXPECT = {'전처리_v1_label.csv': '668a30e4d983ce6ff277a2b2fd41fc96e5f0b4305aea1d550879bfd1f042c861', '전처리_v1_onehot.csv': '2a72de92de2a3c00a53dde6f7679b6fcc4737b9707356acd74358e8f122e333c', '전처리_v2_label.csv': '62d6b7fe6f3a489513a35f08f06404f2683479caab0ec08420da87f65cbece77', '전처리_v2_onehot.csv': '215caf9aad824adfcdaf0796f0d47ac0b33f54b79e218163a72899165dbd9ea3', '1.8_칸설명.csv': '5f38b2322582005d222368da17a718ce29892542c6ac3771293526d91750ebc0'}
for name, h in EXPECT.items():
    same = sha(P.PRE_FINAL / name) == h
    print("같음" if same else "다름", name)
assert all(sha(P.PRE_FINAL / n) == h for n, h in EXPECT.items()), "전처리 결과가 보고서 작성 때와 다르다"

[완료] 1.1_시간복원.py (1초)
    20210715 복원 시각 기온 일치: 기상청 같은 시각 2/24, 한 시간 뒤 24/24
    저장 1_전처리\6_중간산출물\csv\1.1_시간복원.csv (6168, 20)


[완료] 1.2_15분변환.py (1초)
    평균 = floor(네 칸 평균+0.5) 일치: True
    출력 (24672, 14) 시작 2021-01-01 00:00:00 끝 2021-09-14 23:45:00 | 15분 간격 연속, 전력 합 보존


[완료] 1.3_기상교체.py (1초)
    강수량 > 0 시간 수: 532 최대 48.1
    저장 (24672, 14)


[완료] 1.4_달력주야.py (1초)
    평일 공휴일(월~금): 7 일
    저장 (24672, 18)


[완료] 1.5_전력0처리.py (1초)
    전력_채움: 1주 전 같은 시각으로 채운 칸 74 | 채운 값 범위 20.0 ~ 115.0
    저장 (24672, 20)


[완료] 1.6_생산량.py (1초)
    v2 NaN 15분 칸: 192
    저장 (24672, 22)


[완료] 1.7_전기요금.py (1초)
    바뀐 15분 줄: 15840
    저장 (24672, 22)


[완료] 1.8_파일저장.py (1초)
    저장 전처리_v2_label.csv (24672, 21) 생산량 NaN 192
    저장 전처리_v2_onehot.csv (24672, 23) 생산량 NaN 192
같음 전처리_v1_label.csv
같음 전처리_v1_onehot.csv
같음 전처리_v2_label.csv
같음 전처리_v2_onehot.csv
같음 1.8_칸설명.csv


### 1-1. 모델 입력 파일 점검

트리·딥러닝 모델은 팀 공통 형식의 15분 파일(`2_딥러닝/data/timeseries_15min_v1_previous_week.csv`)을 입력으로 쓴다. 이 파일이 위 전처리 확정 파일과 같은 처리를 거쳤는지 칸마다 대조한다. 다른 점은 다음 세 가지다.

- 달력 칸: 우리 파일은 `요일·공휴일·daytype`, 모델 입력은 `day_type`(0 평일, 1 토, 2 일, 3 공휴일). 모델 입력의 `day_type` 3에는 공휴일 10일 외에 8/2~8/6(생산량이 0인 평일 5일)도 들어 있다
- 7/13·7/15 생산량: 모델 입력(v1)은 전주 같은 시각 값
- `공장인원` 칸: 모델 입력에만 있다(딥러닝은 이 칸을 입력으로 쓰지 않는다)

In [3]:
a = pd.read_csv(P.MODEL_INPUT, encoding="utf-8-sig")
r = pd.read_csv(P.PRE_FINAL / "전처리_v1_label.csv", encoding="utf-8-sig")
print("줄 수:", len(a), len(r))
checks = {
    "시각": (pd.to_datetime(a["timestamp"]).values == pd.to_datetime(r["시각"]).values).all(),
    "전력(0 구간은 1주 전 값으로 채움)": (a["power_15min"].values == r["전력_채움"].values).all(),
    "기온·풍속·습도·강수량": all((a[c].values == r[c].values).all() for c in ["기온", "풍속", "습도", "강수량"]),
    "인건비": (a["인건비"].values == r["인건비"].values).all(),
    "전기요금(계절)": (a["전기요금(계절)"].values == r["전기요금_계절"].values).all(),
    "주야(is_night)": (a["is_night"].values == r["is_night"].values).all(),
    "요일(day = 월1~일7)": (a["day"].values == r["요일"].values + 1).all(),
}
# day_type: 우리 달력(요일·공휴일)으로 만든 값과 비교해 다른 날을 보여 준다
ours = r["요일"].map({5: 1, 6: 2}).fillna(0).where(r["공휴일"] == 0, 3).astype(int)
dt_diff = sorted(r.loc[a["day_type"].values != ours.values, "날짜"].unique().tolist())
print("day_type이 우리 달력과 다른 날:", dt_diff, "| 그날들 생산량 합:", float(r.loc[r["날짜"].isin(dt_diff), "생산량"].sum()))
checks["day_type (8/2~8/6만 3으로 다르고 나머지는 같음)"] = dt_diff == [20210802, 20210803, 20210804, 20210805, 20210806]
same_prod = a["생산량"].values == r["생산량"].values
checks["생산량(7/13·7/15 제외)"] = same_prod[~r["날짜"].isin([20210713, 20210715]).values].all()
for k, v in checks.items():
    print("같음" if v else "다름", k)
print("생산량이 다른 날:", sorted(r.loc[~same_prod, "날짜"].unique().tolist()))

줄 수: 24672 24672
day_type이 우리 달력과 다른 날: [20210802, 20210803, 20210804, 20210805, 20210806] | 그날들 생산량 합: 0.0
같음 시각
같음 전력(0 구간은 1주 전 값으로 채움)
같음 기온·풍속·습도·강수량
같음 인건비
같음 전기요금(계절)
같음 주야(is_night)
같음 요일(day = 월1~일7)
같음 day_type (8/2~8/6만 3으로 다르고 나머지는 같음)
같음 생산량(7/13·7/15 제외)
생산량이 다른 날: [20210713, 20210715]


## 2. 딥러닝 (최종 모델)

RNN·LSTM·N-HiTS 세 모델(각 시드 3개 평균)의 단순 평균 앙상블이다. 1~24시간은 피크 가중 손실(C), 4일은 RMSE 손실(A)로 학습한 앙상블을 최종 모델로 썼다. 코드 설명은 `2_딥러닝/README.md`에 있다.

- **기본(재학습 끔):** `main.py --resume`. 저장된 학습 결과(`results/future_known/`)로 7월 선정 → 8월 확인 → 9월 시험과 제출 예측을 다시 만든다. 학습은 하지 않는다.
- **옵션(재학습):** `RUN_DL_RETRAIN = True`이면 `main.py`로 처음부터 학습한다(그리드 768개 + 시드 72개 + 최종 108개, GPU에서 수 시간). 결과는 `results/run_실행시각/`에 따로 저장되고, 장치·torch 버전에 따라 숫자가 조금 달라질 수 있다.

In [4]:
RUN_DL_RETRAIN = False   # True로 바꾸면 처음부터 다시 학습한다 (GPU에서 수 시간)

if RUN_DL_RETRAIN:
    run("main.py", P.DL, show=10)
else:
    run("main.py", P.DL, ["--resume"], show=8, cpu=True)   # 학습하지 않으므로 CPU로 충분하다

# 제출 예측(9월)이 팀이 제출한 파일과 같은지 확인한다 (재학습하면 달라질 수 있다)
EXPECT_TEST = {'12h.csv': '26f98c10e2665e828a20c3155fc59d0069f78268e1ec71021b4627fffe6d1fe2', '1h.csv': '07a63ced7a09c1ecb3b2c541775e9d9dec0a288dd1477cd15563dae765325d97', '24h.csv': '3c5523a017a0d3476a0d27b1d80f974d33cf5447c4da0cd68c6298dcdb90d210', '4d.csv': '5aede2600e90c0607fc42a6f4809c9ec256d6867380be5d3c18543904335f893', '4h.csv': '47230b1cf6dc1b81e31f473f7e76a5862e17c382fc64e707b3b6f750a46ac32e', '8h.csv': '8ed8adfb9442023cc1d9a23ead695bb90ae3e0b56ee991465477aeb87f3ff34d'}
for name, h in EXPECT_TEST.items():
    print("같음" if sha(P.DL / "test_predictions" / name) == h else "다름", "test_predictions/" + name)

[완료] main.py (35초)
    PASS 4h: 10,656행, 발행 시각·실제값·피크 기준·예측값 일치
    PASS 8h: 21,056행, 발행 시각·실제값·피크 기준·예측값 일치
    PASS 12h: 31,200행, 발행 시각·실제값·피크 기준·예측값 일치
    PASS 24h: 60,096행, 발행 시각·실제값·피크 기준·예측값 일치
    PASS 4d: 8,448행, 발행 시각·실제값·피크 기준·예측값 일치
    완료: .\2_딥러닝\test_predictions
    성능 결과: .\2_딥러닝\results_summary.csv
    상세 결과: .\2_딥러닝\results\future_known
같음 test_predictions/12h.csv
같음 test_predictions/1h.csv
같음 test_predictions/24h.csv
같음 test_predictions/4d.csv
같음 test_predictions/4h.csv
같음 test_predictions/8h.csv


In [5]:
# 마진·경보 단계 입력: 대표(앙상블 A·C)와 기준선 4개의 7·8월 예측, 대표의 9월 예측
run("예측_내보내기.py", P.DL, show=8, cpu=True)

[완료] 예측_내보내기.py (21초)
    대표: {'1h A': 'ENSEMBLE A', '1h C': 'ENSEMBLE C', '4h A': 'ENSEMBLE A', '4h C': 'ENSEMBLE C', '8h A': 'ENSEMBLE A', '8h C': 'ENSEMBLE C', '12h A': 'ENSEMBLE A', 
    1h: 7·8월 142,704줄, 9월 10,728줄
    4h: 7·8월 142,272줄, 9월 10,656줄
    8h: 7·8월 283,008줄, 9월 21,056줄
    12h: 7·8월 422,208줄, 9월 31,200줄
    24h: 7·8월 830,592줄, 9월 60,096줄
    4d: 7·8월 129,024줄, 9월 8,448줄


## 3. 트리 모델 (비교 모델)

RF·XGBoost·LightGBM·CatBoost를 피크 가중 손실 C로 학습해 단순 평균한 앙상블이다. 보고서 <표 2-3>의 "트리 앙상블" 열과 <표 2-4>의 비교 모델이다.

- **기본:** 저장 예측(`3_트리모델/저장예측/`, 7·8월)을 종합 오차 지표 C로 채점한다.
- **옵션(재학습):** 트리 학습 코드는 `3_트리모델/학습코드/`에 있다. 실행 환경이 달라서(Python 3.13.5, `3_트리모델/requirements.txt`) 이 노트북 안에서는 실행하지 않고 명령만 보여 준다. 재학습 방법과 주의점은 README "트리 모델 재학습"을 본다.

In [6]:
run("트리_채점.py", ROOT / "3_트리모델", show=8)

[완료] 트리_채점.py (1초)
    달       7월     8월
    구간               
    1h   14.02   9.72
    4h   16.59  14.01
    8h   17.62  16.57
    12h  19.14  20.84
    24h  20.47  35.14
    4d   21.28  34.17


In [7]:
RUN_TREE_RETRAIN = False   # 트리 재학습은 별도 환경(Python 3.13.5)에서 아래 명령으로 실행한다

TREE_CMDS = [
    "python -u -B -X utf8 notebooks/models/random_forest/run_rf_m1007.py --workers 2",
    "python -u -B -X utf8 notebooks/_shared/run_boosting_m1007.py --library xgboost --run-id <run_id>",
    "python -u -B -X utf8 notebooks/_shared/run_boosting_m1007.py --library lightgbm --run-id <run_id>",
    "python -u -B -X utf8 notebooks/_shared/run_boosting_m1007.py --library catboost --run-id <run_id>",
    "python -B -X utf8 notebooks/_shared/m1007_august_rf_run.py rf --rf-run <RF 7월 run 폴더> --out <출력 폴더>",
    "python -B -X utf8 notebooks/_shared/m1007_august_boosting_run.py boost --library xgboost --boost-run <7월 run_id> --out <출력 폴더>",
]
print("트리 재학습 명령 (3_트리모델/학습코드/code 에서 실행, 8월 재학습 = 실행 당시 판 m1007_august_*_run.py):")
for c in TREE_CMDS:
    print("  " + c)

트리 재학습 명령 (3_트리모델/학습코드/code 에서 실행, 8월 재학습 = 실행 당시 판 m1007_august_*_run.py):
  python -u -B -X utf8 notebooks/models/random_forest/run_rf_m1007.py --workers 2
  python -u -B -X utf8 notebooks/_shared/run_boosting_m1007.py --library xgboost --run-id <run_id>
  python -u -B -X utf8 notebooks/_shared/run_boosting_m1007.py --library lightgbm --run-id <run_id>
  python -u -B -X utf8 notebooks/_shared/run_boosting_m1007.py --library catboost --run-id <run_id>
  python -B -X utf8 notebooks/_shared/m1007_august_rf_run.py rf --rf-run <RF 7월 run 폴더> --out <출력 폴더>
  python -B -X utf8 notebooks/_shared/m1007_august_boosting_run.py boost --library xgboost --boost-run <7월 run_id> --out <출력 폴더>


## 4. 마진·경보

딥러닝 앙상블(피크 가중 손실)의 예측으로 안전 마진과 경보를 정하고 8·9월에 확인한다. 코드는 `4_마진경보/5_코드/`, 결과는 `4_마진경보/6_중간산출물/`에 저장된다.

| 코드 | 하는 일 |
|---|---|
| 4.1 | 7월 요금 시간대 피크 근접 칸의 과소예측으로 마진 계산 |
| 4.2 | 그 마진으로 8월 경보 확인 |
| 4.8 | 경보 결과 재채점 (뒤 단계가 공통 함수로 씀) |
| 4.10 | 위험도 순위 경보 |
| 4.10d | 아침 08시 예측의 상위 시간 |
| 4.10e | 위험한 날 판별 (08시 24시간 예측 최고 ÷ b ≥ 기준선 T) |
| 4.13 | R1 마진(7/13·7/15 제외)과 9월 평가 |
| 4.14 | 위험 시간(예측 상위 12칸)과 고정 시간대 비교 |
| 7.8 | 최종 모델과 기준선 4개의 7·8·9월 종합 오차 지표 C |
| 7.9 | 9월 위험한 날·위험 시간 경보 날별 결과 |
| 7.19 | 위험한 날 + 위험 시간 경보를 2026년 요금 체계(07시 발행)로 다시 검증 |

In [8]:
MARGIN_CODE = ROOT / "4_마진경보" / "5_코드"
for s in ["4.1_요금시간마진_저생산피크.py", "4.2_요금시간마진_8월검증.py", "4.8_운영판정_재채점.py",
          "4.10_위험도_순위경보.py", "4.10d_아침예보_상위시간.py", "4.10e_위험한날_판별.py",
          "4.13_R1마진_9월평가.py", "4.14_고정시간_기준선.py", "7.8_9월채점.py", "7.9_9월경보.py",
          "7.19_2026체계경보.py"]:
    run(s, MARGIN_CODE, show=0)

[완료] 4.1_요금시간마진_저생산피크.py (3초)


[완료] 4.2_요금시간마진_8월검증.py (4초)


[완료] 4.8_운영판정_재채점.py (4초)


[완료] 4.10_위험도_순위경보.py (3초)


[완료] 4.10d_아침예보_상위시간.py (2초)


[완료] 4.10e_위험한날_판별.py (2초)


[완료] 4.13_R1마진_9월평가.py (3초)


[완료] 4.14_고정시간_기준선.py (2초)


[완료] 7.8_9월채점.py (26초)


[완료] 7.9_9월경보.py (2초)


[완료] 7.19_2026체계경보.py (2초)


## 5. 피크 분석과 방안

실측 전력에서 피크 사건을 찾고(5장), 목표선과 부하 이동, 요금 환산으로 방안을 만든다(6장).

| 코드 | 하는 일 |
|---|---|
| 5.0~5.7 | 피크 준비·사건 목록·시각과 요일·피크 모양·목표선·생산과 날씨 비교·재가동 |
| 6.1~6.2 | 목표선 후보, 주의 기간 |
| 6.3·6.3b·6.3c | 앞뒤 칸 이동, 목표선 세밀 탐색, 여러 설비 연쇄 조정(선형계획) |
| 6.4~6.5 | 요금 환산, 추천과 점검 |

In [9]:
for stage, names in [("5_피크분석", ["5.0_피크준비.py", "5.1_사건목록.py", "5.2_시각요일.py", "5.3_피크모양.py",
                                    "5.4_목표선.py", "5.5_생산날씨비교.py", "5.7_재가동.py"]),
                     ("6_방안", ["6.1_목표선후보.py", "6.2_주의기간.py", "6.3_앞뒤칸이동.py", "6.3b_목표선세밀.py",
                                "6.3c_선형계획.py", "6.4_요금환산.py", "6.5_추천과점검.py"])]:
    for s in names:
        run(s, ROOT / stage / "5_코드", show=0)

[완료] 5.0_피크준비.py (1초)


[완료] 5.1_사건목록.py (1초)


[완료] 5.2_시각요일.py (1초)


[완료] 5.3_피크모양.py (1초)


[완료] 5.4_목표선.py (2초)


[완료] 5.5_생산날씨비교.py (2초)


... (4 outputs left out)

[완료] 6.3b_목표선세밀.py (2초)


[완료] 6.3c_선형계획.py (29초)


[완료] 6.4_요금환산.py (1초)


[완료] 6.5_추천과점검.py (1초)


## 6. 요약: 보고서 핵심 숫자

위에서 다시 만든 결과 파일에서 보고서의 핵심 숫자를 읽어, 보고서 표 번호와 함께 보여 준다. `보고서` 칸은 결과보고서에 적힌 값이고, `일치` 칸은 둘이 같은지다.

In [10]:
CHECK = []


def show(title, df):
    print("\n" + title)
    print(df.to_string(index=False))
    if "일치" in df:
        CHECK.append((title, bool(df["일치"].all())))


G = ["1h", "4h", "8h", "12h", "24h", "4d"]
s78 = pd.read_csv(ROOT / "4_마진경보/6_중간산출물/csv/7.8_9월채점.csv")
tree = pd.read_csv(ROOT / "3_트리모델/트리_채점.csv")


def c_of(g, month, model, loss, col="공통C"):
    x = s78[(s78["구간"] == g) & (s78["달"] == month) & (s78["모델"] == model) & (s78["손실"] == loss)]
    return float(x[col].iloc[0])


def best_base(g, month, col="공통C"):
    x = s78[(s78["구간"] == g) & (s78["달"] == month) & s78["모델"].str.startswith("기준선")]
    return float(x[col].min())


FINAL_LOSS = {"1h": "C", "4h": "C", "8h": "C", "12h": "C", "24h": "C", "4d": "A"}   # 최종 모델의 손실

# <표 2-3> 7월 종합 오차 지표 C
rep23 = {"1h": (20.63, 14.02, 13.83, 13.42), "4h": (20.54, 16.59, 17.04, 16.52), "8h": (20.60, 17.62, 18.54, 17.50),
         "12h": (20.65, 19.14, 18.70, 18.25), "24h": (20.61, 20.47, 19.31, 18.73), "4d": (22.00, 21.28, 19.51, 20.72)}
rows = []
for g in G:
    v = (best_base(g, "7월"), float(tree[(tree["구간"] == g) & (tree["달"] == "7월")]["공통C"].iloc[0]),
         c_of(g, "7월", "ENSEMBLE", "A"), c_of(g, "7월", "ENSEMBLE", "C"))
    rows.append([g, *v, "/".join(f"{x:.2f}" for x in rep23[g]), all(abs(a - b) < 0.005 for a, b in zip(v, rep23[g]))])
show("<표 2-3> 7월 종합 오차 지표 C (kW): 기준선 · 트리 앙상블 · 딥러닝 A · 딥러닝 C",
     pd.DataFrame(rows, columns=["구간", "기준선", "트리", "딥러닝A", "딥러닝C", "보고서", "일치"]))

# <표 2-5> 최종 모델의 기간별 종합 오차 지표 C, 9월 기준선 대비
rep25 = {"1h": (13.42, 12.95, 9.09, -23.2), "4h": (16.52, 18.09, 10.22, -13.8), "8h": (17.50, 21.93, 10.47, -11.9),
         "12h": (18.25, 25.07, 10.34, -13.1), "24h": (18.73, 32.64, 10.48, -11.3), "4d": (19.51, 38.21, 10.27, -9.8)}
rows = []
for g in G:
    f = [c_of(g, m, "ENSEMBLE", FINAL_LOSS[g]) for m in ["7월", "8월", "9월"]]
    gap = round((f[2] / best_base(g, "9월") - 1) * 100, 1)
    v = (*f, gap)
    rows.append([g, *v, round((f[1] / best_base(g, "8월") - 1) * 100, 1), all(abs(a - b) < 0.05 for a, b in zip(v, rep25[g]))])
show("<표 2-5> 최종 모델 종합 오차 지표 C (kW) 7·8·9월, 9월 기준선 대비(%), 8월 기준선 대비(%)",
     pd.DataFrame(rows, columns=["구간", "7월", "8월", "9월", "9월 대비%", "8월 대비%", "일치"]))

# <표 2-6> 9월 RMSE
rep26 = {"1h": 8.62, "4h": 9.61, "8h": 9.94, "12h": 9.83, "24h": 9.99, "4d": 10.08}
rows = [[g, c_of(g, "9월", "ENSEMBLE", FINAL_LOSS[g], "RMSE"), c_of(g, "9월", "기준선 지난주", "-", "RMSE"), rep26[g],
         abs(c_of(g, "9월", "ENSEMBLE", FINAL_LOSS[g], "RMSE") - rep26[g]) < 0.005] for g in G]
show("<표 2-6> 9월 RMSE (kW)", pd.DataFrame(rows, columns=["구간", "최종 모델", "전주 같은 시각", "보고서", "일치"]))

# 기본 마진 R1 (7월 요금 시간대 피크 근접 칸, 7/13·7/15 제외, 과소예측의 95% 지점)
r13 = pd.read_csv(ROOT / "4_마진경보/6_중간산출물/csv/4.13_R1마진_9월평가.csv")
r1 = r13[r13["규칙"].str.contains("R1")]
repR1 = {"1h": 11.60, "4h": 14.09, "8h": 15.28, "12h": 15.26, "24h": 15.42}
rows = [[g, round(float(x["마진"]) * 100, 2), int(x["팀_피크근처_날"]), int(x["팀_잡은날"]), round(float(x["팀_하루경보시간(h)"]), 1),
         repR1[g], abs(float(x["마진"]) * 100 - repR1[g]) < 0.005] for g, x in zip(r1["구간"], r1.to_dict("records"))]
show("기본 마진 R1 (b 대비 %)과 9월 결과: 피크 근접 날 · 잡은 날 · 하루 경보 시간",
     pd.DataFrame(rows, columns=["구간", "R1 마진%", "9월 피크근접날", "잡은날", "하루경보h", "보고서", "일치"]))

# <표 3-2> 위험 시간 경보 (위험한 날 + 예측 상위 15분 12칸)
t14 = pd.read_csv(ROOT / "4_마진경보/6_중간산출물/csv/4.14_고정시간_기준선.csv")
rep32 = {"7월": (17, 17, 14), "8월": (6, 13, 5), "9월": (9, 9, 8)}
rows = [[m, int(x["피크근접_날"]), int(x["알린_날"]), int(x["상위12칸_잡은날"]), int(x["고정시간_잡은날"]),
         rep32[m] == (int(x["피크근접_날"]), int(x["알린_날"]), int(x["상위12칸_잡은날"]))] for m, x in zip(t14["달"], t14.to_dict("records"))]
show("<표 3-2> 위험 시간 경보: 피크 근접 날 · 알린 날 · 위험 시간으로 잡은 날 (참고: 고정 09~12시로 잡은 날)",
     pd.DataFrame(rows, columns=["기간", "피크근접날", "알린날", "잡은날", "고정시간_잡은날", "일치"]))

# <표 4-1> 조정 방식별 결과 (2026년 요금 체계, 덜어낼 비율 10%)
fee = pd.read_csv(ROOT / "6_방안/3_최종데이터/6.4_요금환산.csv")
fee = fee[(fee["판"] == 2026) & (fee["f"] == 0.1)]
lp = pd.read_csv(ROOT / "6_방안/3_최종데이터/6.3c_선형계획.csv")
lp = lp[(lp["판"] == 2026) & (lp["f"] == 0.1)]
rows = []
for name, target, k, rep in [("한 설비 15분 (목표 212)", 212, 1, (212.0, 866400)),
                             ("한 설비 30분 (목표 202)", 202, 2, (211.0, 953040)),
                             ("한 설비 60분 (목표 202)", 202, 4, (202.0, 1732800))]:
    x = fee[(fee["P"] == target) & (fee["k"] == k)].iloc[0]
    v = (float(x["남은최고_별"]), int(x["기본절감_1년_선택I"]))
    rows.append([name, *v, rep, v == rep])
for name, target, rep in [("여러 설비 연쇄 15분 (목표 202)", "202", (202.0, 1732800)),
                          ("여러 설비 연쇄 (가능한 최저)", "최저(199.8)", (199.8, 1923408))]:
    x = lp[lp["목표P"].astype(str) == target].iloc[0]
    v = (float(x["남은최고_별"]), int(x["기본절감_1년_선택I"]))
    rows.append([name, *v, rep, v == rep])
show("<표 4-1> 조정 방식별 결과 (2026년 요금 체계, 덜어낼 비율 10%): 이동 후 요금적용전력(kW) · 연간 기본요금 절감(원, 선택 I)",
     pd.DataFrame(rows, columns=["조정", "요금적용전력", "연간절감", "보고서", "일치"]))

print("\n=== 대조 결과 ===")
for t, ok in CHECK:
    print("일치" if ok else "불일치", "|", t)


<표 2-3> 7월 종합 오차 지표 C (kW): 기준선 · 트리 앙상블 · 딥러닝 A · 딥러닝 C
 구간   기준선    트리  딥러닝A  딥러닝C                     보고서   일치
 1h 20.63 14.02 13.83 13.42 20.63/14.02/13.83/13.42 True
 4h 20.54 16.59 17.04 16.52 20.54/16.59/17.04/16.52 True
 8h 20.60 17.62 18.54 17.50 20.60/17.62/18.54/17.50 True
12h 20.65 19.14 18.70 18.25 20.65/19.14/18.70/18.25 True
24h 20.61 20.47 19.31 18.73 20.61/20.47/19.31/18.73 True
 4d 22.00 21.28 19.51 20.72 22.00/21.28/19.51/20.72 True

<표 2-5> 최종 모델 종합 오차 지표 C (kW) 7·8·9월, 9월 기준선 대비(%), 8월 기준선 대비(%)
 구간    7월    8월    9월  9월 대비%  8월 대비%   일치
 1h 13.42 12.95  9.09   -23.2   -38.1 True
 4h 16.52 18.09 10.22   -13.8   -28.7 True
 8h 17.50 21.93 10.47   -11.9   -14.8 True
12h 18.25 25.07 10.34   -13.1    -3.8 True
24h 18.73 32.64 10.48   -11.3    22.7 True
 4d 19.51 38.21 10.27    -9.8    29.0 True

<표 2-6> 9월 RMSE (kW)
 구간  최종 모델  전주 같은 시각   보고서   일치
 1h   8.62     11.19  8.62 True
 4h   9.61     11.21  9.61 True
 8h   9.94     11.25  9.94 True
12h   9.83     11.25  9.83

## 7. 결과 모으기

제출 예측결과 파일(9월 시험 예측)과 요약표를 `결과/`에 복사한다.

In [11]:
dst = P.RESULT / "9월_시험예측"
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(P.DL / "test_predictions", dst)
shutil.copy2(P.DL / "results_summary.csv", P.RESULT / "성능요약_딥러닝.csv")
shutil.copy2(ROOT / "4_마진경보/6_중간산출물/csv/7.8_9월채점.csv", P.RESULT / "종합오차C_7_8_9월.csv")
shutil.copy2(ROOT / "3_트리모델/트리_채점.csv", P.RESULT / "종합오차C_트리_7_8월.csv")
shutil.copy2(ROOT / "4_마진경보/6_중간산출물/csv/4.13_R1마진_9월평가.csv", P.RESULT / "R1마진_9월평가.csv")
shutil.copy2(ROOT / "4_마진경보/6_중간산출물/csv/7.9_9월경보_날별.csv", P.RESULT / "9월경보_날별.csv")
shutil.copy2(ROOT / "6_방안/3_최종데이터/6.5_추천.csv", P.RESULT / "방안_추천.csv")
for p in sorted(P.RESULT.rglob("*.csv")):
    print(p.relative_to(ROOT))

결과\9월_시험예측\12h.csv
결과\9월_시험예측\1h.csv
결과\9월_시험예측\24h.csv
결과\9월_시험예측\4d.csv
결과\9월_시험예측\4h.csv
결과\9월_시험예측\8h.csv
결과\9월경보_날별.csv
결과\R1마진_9월평가.csv
결과\방안_추천.csv
결과\성능요약_딥러닝.csv
결과\종합오차C_7_8_9월.csv
결과\종합오차C_트리_7_8월.csv
